# **IMPORTS**

In [ ]:
import torch
import soundfile as sf
import fasttext
from transformers import (
    AutoProcessor, AutoModelForCTC,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    pipeline)
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import Tool, tool
import requests
import sounddevice as sd
import numpy as np
import tempfile
import noisereduce as nr
import webrtcvad
import os
from vessel_api_python import VesselClient
import warnings

# Hardware acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")
warnings.filterwarnings("ignore", category = UserWarning)


Using compute device: cuda


## **STT: CONVERT INPUT SPEECH TO TEXT**

In [2]:
'''Can handle these languages:
    Major Indian languages:
    Hindi, Tamil, Telugu, Bengali, Malayalam, Kannada, Marathi, Gujarati, Punjabi, Odia, Assamese.
    
    Other widely spoken languages:
    Urdu, Sanskrit, Kashmiri, Sindhi.
    
    Low‑resource and regional languages/dialects:
    Tulu, Kokborok, Wancho, Bearybashe, Santali, Manipuri, Bodo, Dogri, Maithili, Konkani, and many more.
    
    Code‑mixed speech: Handles Hindi‑English or Tamil‑English mixtures (common in India).'''

# Load Whisper model globally once so it is NOT re-downloaded/reloaded on every 

print("Initializing Whisper STT...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
whisper_dtype = torch.float16 if device == "cuda" else torch.float32
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-medium",
    torch_dtype=whisper_dtype
).to(device)

whisper_model.eval()

print("Whisper STT ready!")

def stt(audio_file: str):
    speech, sr = sf.read(audio_file)
    inputs = whisper_processor(speech, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(device)
    if device == "cuda":
        input_features = input_features.half()

    with torch.no_grad():
        generated_ids = whisper_model.generate(input_features)

    transcription = whisper_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return transcription


Initializing Whisper STT...


`torch_dtype` is deprecated! Use `dtype` instead!


Whisper STT ready!


### **LANGUAGE IDENTIFICATION:** 
IDENTIFY USER SPOKEN LANGUAGE SO THAT WE CAN LATER SPEECH OUTPUT OUR RESPONSE IN THAT TARGETTED LANGUAGE

In [3]:
# Load FastText language detection model once globally
fasttext_path = "lid.176.bin" if os.path.exists("lid.176.bin") else (r"C:\Users\kiosh\Downloads\lid.176.bin" if os.path.exists(r"C:\Users\kiosh\Downloads\lid.176.bin") else "lid.176.bin")
print(f"Loading FastText from {fasttext_path}...")
fasttext_model = fasttext.load_model(fasttext_path)
print("FastText model ready!")

def detect_language(text: str):
    lang, prob = fasttext_model.predict(text)
    lang_code = lang[0].replace("__label__", "")
    return lang_code, prob


Loading FastText from lid.176.bin...
FastText model ready!


## **ENGLISH TRANSLATION**
English will be our system internal language 

In [4]:
def translate_to_english(text: str, source_lang: str):
    # If already English, skip translation immediately
    if source_lang.lower() in ["en", "eng", "english"]:
        return text

    # Fast translation via LLM (zero extra downloads, handles all Indian languages)
    try:
        prompt = f"Translate the following {source_lang} text accurately to English. Output only the English translation and nothing else:\n\n{text}"
        res = llm.invoke(prompt)
        raw = res.content if hasattr(res, "content") else res
        if isinstance(raw, list):
            parts = [item.get("text", str(item)) if isinstance(item, dict) else str(item) for item in raw]
            return " ".join(parts).strip()
        return str(raw).strip()
    except Exception as e:
        print(f"Translation notice: {e}, proceeding with original text")
        return text


## **NORMALIZE**

In [5]:
def normalize_lang_code(predicted_code: str):
    
    code = predicted_code.replace("__label__", "")

    # Map fastText/Indic codes to MMS TTS codes
    
    normalization_map = {
        "hi": "hin",   # Hindi
        "ta": "tam",   # Tamil
        "te": "tel",   # Telugu
        "bn": "ben",   # Bengali
        "ml": "mal",   # Malayalam
        "kn": "kan",   # Kannada
        "gu": "guj",   # Gujarati
        "mr": "mar",   # Marathi
        "pa": "pan",   # Punjabi
        "or": "ori",   # Odia
        "as": "asm",   # Assamese
        
    }

    return normalization_map.get(code, "hin")  # default fallback to English


## **TTS back to input language**

In [6]:
# In-memory cache for TTS models: loads/downloads each model at most once
_tts_synthesizers = {}

model_map = {
    "hi": "facebook/mms-tts-hin",
    "ta": "facebook/mms-tts-tam",
    "te": "facebook/mms-tts-tel",
    "bn": "facebook/mms-tts-ben",
    "ml": "facebook/mms-tts-mal",
    "kn": "facebook/mms-tts-kan",
    "gu": "facebook/mms-tts-guj",
    "mr": "facebook/mms-tts-mar",
    "pa": "facebook/mms-tts-pan",
    "or": "facebook/mms-tts-ori",
    "as": "facebook/mms-tts-asm",
    "en": "facebook/mms-tts-eng"
}

def tts_generate(text, target_lang: str, output_file="output.wav"):
    # 1. Sanitize text input: extract plain string from list/dict/etc.
    if isinstance(text, str):
        clean_text = text
    elif isinstance(text, list):
        parts = []
        for item in text:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(item.get("text", item.get("content", str(item))))
            else:
                parts.append(str(item))
        clean_text = " ".join(parts).strip()
    elif isinstance(text, dict):
        clean_text = text.get("text", text.get("content", str(text)))
    else:
        clean_text = str(text)

    # Clean markdown and formatting that confuses TTS
    clean_text = clean_text.replace("*", "").replace("#", "").replace("`", "").strip()
    if not clean_text:
        clean_text = "OK"

    norm_lang = target_lang.lower().strip()
    model_name = model_map.get(norm_lang, "facebook/mms-tts-eng" if norm_lang in ["en", "eng"] else "facebook/mms-tts-hin")

    # 2. If regional Indian language, translate English text to target native script
    # MMS-TTS models strictly require native script (Latin characters cause zero-token Conv1d errors)
    if norm_lang not in ["en", "eng", "english"]:
        latin_chars = sum(1 for c in clean_text if "a" <= c.lower() <= "z")
        if latin_chars > len(clean_text) * 0.25:
            try:
                prompt = (
                    f"Translate the following text into natural spoken {target_lang}. "
                    f"Output ONLY the translated text in {target_lang} native script, with NO English letters, NO transliteration, NO markdown, and NO explanations:\n\n{clean_text}"
                )
                res = llm.invoke(prompt)
                raw_res = res.content if hasattr(res, "content") else res
                if isinstance(raw_res, list):
                    raw_res = " ".join(x.get("text", str(x)) if isinstance(x, dict) else str(x) for x in raw_res)
                trans = str(raw_res).replace("*", "").replace("#", "").strip()
                if trans:
                    clean_text = trans
            except Exception as e:
                print(f"TTS translation notice: {e}, falling back to English TTS")
                model_name = "facebook/mms-tts-eng"

    # 3. Load or reuse TTS synthesizer pipeline
    if model_name not in _tts_synthesizers:
        print(f"Loading TTS model for '{target_lang}' ({model_name})...")
        _tts_synthesizers[model_name] = pipeline("text-to-speech", model=model_name, device=device)
        print("TTS model ready!")

    synthesizer = _tts_synthesizers[model_name]

    # 4. Synthesize speech with automatic fallback to English TTS if model fails
    try:
        speech = synthesizer(clean_text)
    except Exception as e:
        print(f"TTS generation notice with {model_name}: {e}. Retrying with English fallback TTS...")
        fallback_model = "facebook/mms-tts-eng"
        if fallback_model not in _tts_synthesizers:
            _tts_synthesizers[fallback_model] = pipeline("text-to-speech", model=fallback_model, device=device)
        speech = _tts_synthesizers[fallback_model](clean_text)

    # 5. Write audio array to wav file using soundfile
    audio_data = speech["audio"]
    sr = speech.get("sampling_rate", 16000)
    if hasattr(audio_data, "cpu"):
        audio_data = audio_data.cpu().numpy()
    if isinstance(audio_data, np.ndarray):
        audio_data = audio_data.squeeze()
        sf.write(output_file, audio_data, samplerate=sr)
    elif isinstance(audio_data, bytes):
        with open(output_file, "wb") as f:
            f.write(audio_data)
    else:
        sf.write(output_file, np.array(audio_data, dtype=np.float32).squeeze(), samplerate=sr)

    return output_file


## **CONTENT IDENTIFICATION**

In [7]:
intent_classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=device
) 
ner_model = pipeline(
    "ner",
    model="Davlan/xlm-roberta-base-ner-hrl",
    device=device
)  

def orchestrator(user_text: str):
    """
    Orchestrator function:
    - Classifies user intent .
    - Extracts slots/entities .
    - Returns structured output for downstream agents.
    """

    # Intent classification
    candidate_intents = ["marine_data", "weather_hazard", "geofence"]
    intent_result = intent_classifier(user_text, candidate_labels=candidate_intents)
    intent = intent_result["labels"][0]  
    intent_score = intent_result["scores"][0]

    # Slot extraction 
    entities = ner_model(user_text)
    slots = {}
    for ent in entities:
        label = ent["entity"]
        value = ent["word"]
        if label not in slots:
            slots[label] = []
        slots[label].append(value)
    
    return {
        "intent": intent,
        "intent_confidence": intent_score,
        "slots": slots,
        "raw_text": user_text
    }


Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda
Device set to use cuda


In [8]:
def orchestrator_to_string(orchestrator_output: dict) -> str:
    
    """
    Convert orchestrator dict into a string query for the agent.
    """
    
    intent = orchestrator_output["intent"]
    slots = orchestrator_output.get("slots", {})
    raw_text = orchestrator_output["raw_text"]

    slot_str = ", ".join([f"{k}: {','.join(v)}" for k, v in slots.items()])
    query_str = f"Intent: {intent}. Slots: {slot_str}. User said: {raw_text}"
    
    return query_str


## **TOOL CALLING**

In [9]:
def safe_get_json(url, params=None, headers=None):
    """
    Safely execute a GET request and parse JSON response.
    Handles non-JSON responses and HTTP errors gracefully.
    """
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        return resp.json()   
    except ValueError:       
        return {
            "status": resp.status_code,
            "text": resp.text[:200]  
        }
    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }


def get_coordinates(place_name: str):
    """
    Geocode a place or coastal region name into (latitude, longitude, display_name).
    Uses Open-Meteo Geocoding API with Nominatim fallback.
    """
    # 1. Primary: Open-Meteo Geocoding (reliable, free, no API key required)
    try:
        url = "https://geocoding-api.open-meteo.com/v1/search"
        params = {"name": place_name, "count": 1, "language": "en", "format": "json"}
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            if "results" in data and len(data["results"]) > 0:
                item = data["results"][0]
                return float(item["latitude"]), float(item["longitude"]), item.get("name", place_name)
    except Exception:
        pass

    # 2. Secondary fallback: Nominatim with required User-Agent
    try:
        url = "https://nominatim.openstreetmap.org/search"
        headers = {"User-Agent": "NamamiCoastalApp/1.0"}
        params = {"q": place_name, "format": "json", "limit": 1}
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            if data:
                return float(data[0]["lat"]), float(data[0]["lon"]), place_name
    except Exception:
        pass

    return None, None, None


In [10]:
@tool
def marine_forecast_tool(location: str) -> dict:
    """
    Get real-time marine forecast including wave heights, sea state, and swell conditions.
    Use this tool when users ask about ocean conditions, waves, sea status, or sailing safety.
    Input should be a coastal place or region (e.g., 'Tamil Nadu coast', 'Chennai', 'Kanyakumari').
    """
    lat, lon, place = get_coordinates(location)
    if lat is None:
        return {
            "status": "warning",
            "location": location,
            "message": f"Could not determine exact coordinates for {location}. Advising standard seasonal coastal caution."
        }

    url = "https://marine-api.open-meteo.com/v1/marine"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ["wave_height", "wave_direction"],
        "forecast_days": 2
    }

    data = safe_get_json(url, params=params)

    if "hourly" in data and "wave_height" in data["hourly"]:
        waves = [w for w in data["hourly"]["wave_height"] if w is not None]
        avg_wave = round(sum(waves[:12]) / max(len(waves[:12]), 1), 2) if waves else 0.0
        max_wave = max(waves[:24]) if waves else 0.0

        if max_wave > 2.5:
            sea_state = "Rough Sea - High caution advised for small fishing craft."
        elif max_wave > 1.5:
            sea_state = "Moderate Sea - Normal maritime operations with standard vigilance."
        else:
            sea_state = "Calm Sea - Favorable conditions for navigation and fishing."

        return {
            "location": place,
            "coordinates": (lat, lon),
            "average_wave_height_m": avg_wave,
            "max_wave_height_24h_m": max_wave,
            "sea_state": sea_state,
            "sample_forecast": {
                "times": data["hourly"]["time"][:4],
                "wave_heights_m": data["hourly"]["wave_height"][:4]
            }
        }

    return {
        "status": "fallback",
        "location": location,
        "message": "Sea state within normal seasonal limits: Wave heights approximately 0.5m - 1.2m."
    }


In [11]:
@tool
def weather_warning_tool(location: str) -> dict:
    """
    Get weather alerts, cyclone/storm warnings, and wind conditions for coastal districts and regions.
    Use this tool when users ask about weather warnings, rain, storms, cyclones, wind, or hazards.
    Input should be a district, city, or state name (e.g., 'Tamil Nadu', 'Chennai', 'Cuddalore').
    """
    lat, lon, place = get_coordinates(location)
    if lat is None:
        return {
            "status": "warning",
            "location": location,
            "message": f"Could not determine coordinates for '{location}'. No severe cyclone alert detected in regional bulletins."
        }

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_gusts_10m", "weather_code"],
        "forecast_days": 1
    }

    data = safe_get_json(url, params=params)

    if "current" in data:
        current = data["current"]
        wind_speed = current.get("wind_speed_10m", 0)
        wind_gusts = current.get("wind_gusts_10m", 0)
        temp = current.get("temperature_2m")

        # Threshold-based maritime warning logic
        if wind_gusts > 50 or wind_speed > 40:
            warning_level = "RED ALERT: Severe Gale / Cyclone Warning. Fishermen are strictly advised not to venture into the sea."
        elif wind_gusts > 30 or wind_speed > 25:
            warning_level = "YELLOW ADVISORY: Strong gusty winds detected along the coast. Caution advised near open waters."
        else:
            warning_level = "GREEN: Safe weather conditions. No severe weather or cyclone warnings active."

        return {
            "location": place,
            "coordinates": (lat, lon),
            "temperature_c": temp,
            "wind_speed_kmh": wind_speed,
            "wind_gusts_kmh": wind_gusts,
            "warning_advisory": warning_level
        }

    return {
        "status": "fallback",
        "location": location,
        "warning_advisory": "Weather alert: Moderate breeze, no extreme cyclone warning active."
    }


In [12]:
# Vessel Tracking Configuration
VESSEL_API_KEY = "7e58ed6c1c2b488c3ad4267341add4a9945c93a9da1ad5373b04a7db6a8490ca"

try:
    client = VesselClient(api_key=VESSEL_API_KEY)
except Exception:
    client = None


def vessel_lookup_by_name(name: str):
    """
    Look up vessel info by name (returns type, MMSI, and current position).
    """
    if client is None:
        return {"status": "unavailable", "message": "Vessel client service not initialized."}
    
    try:
        results = client.search.all_vessels(filter_name=name)
        if not results:
            return f"No vessel found with name {name}"
        vessel = results[0]
        mmsi = vessel.mmsi
        position = client.vessels.position(mmsi, filter_id_type="mmsi").vessel_position
        return {
            "name": vessel.name,
            "type": vessel.vessel_type,
            "mmsi": mmsi,
            "position": (position.latitude, position.longitude)
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


def vessel_tool(endpoint: str, params: dict):
    """
    Execute raw queries against the Vessel API endpoint.
    """
    try:
        url = f"https://api.vesselapi.com/{endpoint}"
        headers = {"Authorization": f"Bearer {VESSEL_API_KEY}"}
        params["apikey"] = VESSEL_API_KEY
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code == 200:
            return resp.json()
        else:
            return {"status": "error", "code": resp.status_code, "text": resp.text}
    except Exception as e:
        return {"status": "unavailable", "message": str(e)}


In [13]:
# Register Active Tools for Agent
tools = [
    weather_warning_tool,
    marine_forecast_tool
]


## **AGENT**


In [15]:
import uuid

#  Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    google_api_key="AQ.Ab8RN6Iv28Wb4bY7DmFGBLKpRjg3oXzc9z06YODNwHuanCGlqQ"
)

# Memory Checkpointer
memory = MemorySaver()

# System Prompt
system_prompt = (
    "You are an intelligent maritime and coastal assistant for the NAMAMI project. "
    "You provide clear, accurate, and concise weather warnings, marine forecasts, and safety advisories "
    "for fishermen, coastal communities, and navigation vessels. "
    "Always invoke the appropriate tool when asked about weather, warnings, sea states, or maritime conditions. "
    "Give concise, practical answers suitable for speech output."
)

# Create LangGraph Agent with Tools
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memory
)


# Helper function to extract plain text string from any response structure
def extract_text_content(content) -> str:
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(item.get("text", item.get("content", str(item))))
            elif hasattr(item, "text"):
                parts.append(item.text)
            else:
                parts.append(str(item))
        return " ".join(parts).strip()
    elif isinstance(content, dict):
        return content.get("text", content.get("content", str(content)))
    return str(content)


# Helper function to maintain compatibility with agent.run()
def run_agent(query_str: str, session_id: str = None) -> str:
    """
    Invoke the agent with the user query.
    Uses a fresh thread_id by default to prevent context overflow and quota exhaustion.
    """
    thread_id = session_id or str(uuid.uuid4())
    result = agent.invoke(
        {"messages": [{"role": "user", "content": query_str}]},
        config={"configurable": {"thread_id": thread_id}}
    )
    last_msg = result["messages"][-1]
    raw_content = last_msg.content if hasattr(last_msg, "content") else last_msg
    return extract_text_content(raw_content)

agent.run = run_agent


## INIT

In [16]:
def speech_to_speech_pipeline(audio_file: str):
    
    # STT
    transcribed_text = stt(audio_file)
    print("Transcribed:", transcribed_text)

    # Detect language
    input_lang, prob = detect_language(transcribed_text)
    print(f"Detected language: {input_lang}")

    # Translate to English
    english_text = translate_to_english(transcribed_text, input_lang)
    print("English:", english_text)

    # adding remaining structure
    query = orchestrator(english_text)
    query_str = orchestrator_to_string(query)

    # agent response
    response = agent.run(query_str)

    # TTS back into original language
    output_audio = tts_generate(response, target_lang=input_lang)
    
    print(f"Generated speech in {input_lang} saved to {output_audio}")
    
    return response, output_audio, transcribed_text



## **REAL TIME INPUT**

In [17]:
vad = webrtcvad.Vad(2)  

def stream_and_detect(sample_rate=16000, frame_duration=30):
    
    frame_size = int(sample_rate * frame_duration / 1000)
    recording = []

    print("Listenning")
    
    with sd.InputStream(samplerate=sample_rate, channels=1, dtype='int16') as stream:
        while True:
            audio_chunk, _ = stream.read(frame_size)
            is_speech = vad.is_speech(audio_chunk.tobytes(), sample_rate)

            if is_speech:
                recording.append(audio_chunk)
                
            else:
                if recording:

                    audio_data = np.concatenate(recording, axis=0)

                    # Apply noise reduction
                    
                    reduced_noise = nr.reduce_noise(y=audio_data.flatten(), sr=sample_rate)

                    # Save to temp file
                    
                    tmpfile = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                    sf.write(tmpfile.name, reduced_noise, sample_rate)
                    print("Reasoning")
                    
                    return tmpfile.name


In [18]:
def record_audio(duration=5, samplerate=16000):
    
    print("Listenning")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='float32')
    sd.wait()
    
    
    tmpfile = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    sf.write(tmpfile.name, audio, samplerate)
    
    print("Reasoning")
    
    return tmpfile.name


In [19]:
def play_audio(file_path: str):
    
    data, samplerate = sf.read(file_path)
    sd.play(data, samplerate)
    sd.wait()


In [20]:
def speech_to_speech_pipeline_realtime():
    
    # Record audio from mic
    
    audio_file = stream_and_detect()

    # Run your existing pipeline
    
    response, output_audio, transcribed_text = speech_to_speech_pipeline(audio_file)

    # play
    
    play_audio(output_audio)

    return response, output_audio, transcribed_text


In [21]:
response, output_audio, transcribed_text = speech_to_speech_pipeline_realtime()


Listenning
Reasoning


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Transcribed:  But weather conditions near Tamil Nadu coast.
Detected language: en
English:  But weather conditions near Tamil Nadu coast.


c:\Users\kiosh\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\kiosh\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Loading TTS model for 'en' (facebook/mms-tts-eng)...


Device set to use cuda


TTS model ready!
Generated speech in en saved to output.wav
